In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
train_df.shape, test_df.shape

((8693, 14), (4277, 13))

In [4]:
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
train_df.shape, test_df.shape

((8693, 14), (4277, 13))

In [6]:
def prepare_features(df, is_train=True):
    # Remove ID and Name columns
    if is_train:
        features = df.drop(columns=['PassengerId', 'Name', 'Transported'])
        target = df['Transported']
    else:
        features = df.drop(columns=['PassengerId', 'Name'])
        target = None
    
    return features, target

In [7]:

X_train_full, y_train_full = prepare_features(train_df, is_train=True)
X_test_full, _ = prepare_features(test_df, is_train=False)

In [8]:
# Handle categorical variables consistently
categorical_columns = X_train_full.select_dtypes(include=['object']).columns
numerical_columns = X_train_full.select_dtypes(exclude=['object']).columns

print(f"Categorical columns: {list(categorical_columns)}")
print(f"Numerical columns: {list(numerical_columns)}")

Categorical columns: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']
Numerical columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


In [9]:
# Handle missing values
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# Fit imputers on training data
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_full[numerical_columns]), # impute numerical columns
    columns=numerical_columns, # columns for numerical data
    index=X_train_full.index # index to maintain original row indices
)
X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train_full[categorical_columns]), 
    columns=categorical_columns,
    index=X_train_full.index
)

# Apply same imputation to test data
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_full[numerical_columns]), 
    columns=numerical_columns,
    index=X_test_full.index
)
X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test_full[categorical_columns]), 
    columns=categorical_columns,
    index=X_test_full.index
)

In [10]:
# 6. Encode categorical variables consistently
label_encoders = {}
for column in categorical_columns:
    le = LabelEncoder()
    # Fit on training data
    le.fit(X_train_cat[column])
    label_encoders[column] = le
    
    # Transform training data
    X_train_cat[column] = le.transform(X_train_cat[column])
    
    # Transform test data, handling unseen labels
    test_values = X_test_cat[column]
    # Map unseen labels to the most frequent class (index 0)
    test_values_mapped = test_values.apply(
        lambda x: x if x in le.classes_ else le.classes_[0]
    )
    X_test_cat[column] = le.transform(test_values_mapped)

In [11]:
# 7. Combine numerical and categorical features
X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_cat], axis=1)

print(f"Training data shape: {X_train_processed.shape}")
print(f"Test data shape: {X_test_processed.shape}")

Training data shape: (8693, 11)
Test data shape: (4277, 11)


In [12]:
# 8. Train model with proper validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_processed, y_train_full, test_size=0.2, random_state=42
)

# Train the model
# rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# rf_model.fit(X_train, y_train)

In [13]:
# # Hyperparameter tuning can be done here if needed
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 10, 20],
#     'min_samples_split': [2, 5, 10]
# }
# grid_search = GridSearchCV(rf_model, param_grid, cv=3, scoring='accuracy')
# grid_search.fit(X_train, y_train)


In [14]:
# # apply the best model
# rf_model = grid_search.best_estimator_


In [15]:
# # Validate the model
# y_val_pred = rf_model.predict(X_val)
# val_accuracy = accuracy_score(y_val, y_val_pred)
# print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

In [16]:
# applying other models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV


models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Gradient Boosting': GradientBoostingClassifier(),
    'SVM': SVC(probability=True),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}
# Creating hyperparameter grids for each model
# param_grids = {
#     'Logistic Regression': {
#         'C': [0.01, 0.1, 1, 10],
#         'solver': ['liblinear', 'saga']
#     },
#     'Gradient Boosting': {
#         'n_estimators': [50, 100],
#         'learning_rate': [0.01, 0.1],
#         'max_depth': [3, 5]
#     },
#     'SVM': {
#         'C': [0.1, 1, 10],
#         'kernel': ['linear', 'rbf']
#     },
#     'XGBoost': {
#         'n_estimators': [50, 100],
#         'learning_rate': [0.01, 0.1],
#         'max_depth': [3, 5]
#     }
# }

# hyperparameter tuning for xgboost
# param_grid_xgb = {
#     'n_estimators': [50, 100],
#     'learning_rate': [0.01, 0.1],
#     'max_depth': [3, 5]
# }
# xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
# grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=3, scoring='accuracy')
# grid_search_xgb.fit(X_train, y_train)
# # Apply the best XGBoost model
# xgb_model_best = grid_search_xgb.best_estimator_
# # Make predictions on test set
# test_predictions = xgb_model_best.predict(X_val)



In [17]:
# applying gradient boosting with hyperparameter tuning
param_grid_gb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5]
}
gb_model = GradientBoostingClassifier()
grid_search_gb = GridSearchCV(gb_model, param_grid_gb, cv=3, scoring='accuracy')
grid_search_gb.fit(X_train, y_train)
# Apply the best Gradient Boosting model
gb_model_best = grid_search_gb.best_estimator_
# Make predictions on validation set
y_val_pred_gb = gb_model_best.predict(X_val)
val_accuracy_gb = accuracy_score(y_val, y_val_pred_gb)
print(f"Gradient Boosting Validation Accuracy: {val_accuracy_gb * 100:.2f}%")   


Gradient Boosting Validation Accuracy: 79.64%


In [18]:
# # accuracy for xgboost
# accuracy_xgb = accuracy_score(y_val, test_predictions)
# print(f"XGBoost Validation Accuracy: {accuracy_xgb * 100:.2f}%")


In [19]:
# Make predictions on test set
test_predictions_final = xgb_model_best.predict(X_test_processed)

NameError: name 'xgb_model_best' is not defined

In [20]:
# make predictions for gradient boosting
test_predictions_gb = gb_model_best.predict(X_test_processed).astype(bool)
print(test_predictions_gb)


[ True False  True ...  True  True  True]


In [ ]:
# for xgboost convert predictions to true/false, 1 is True, 0 is False
test_predictions_final = test_predictions_final.astype(bool)
print(test_predictions_final)

[ True False  True ...  True  True  True]


In [22]:
# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': test_predictions_gb
})

submission.to_csv('submission_fixed.csv', index=False)
print(f"Predictions saved to 'submission_fixed.csv'")
print(f"Number of predictions: {len(submission)}")
submission.head()

Predictions saved to 'submission_fixed.csv'
Number of predictions: 4277


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
